# 08 — ML (eye-tracking)

Modelo unimodal de eye-tracking do artigo: **ensemble de Gradient Boosting** sobre as 252
features do `03_Features_et`, avaliado por **10-fold estratificado repetido 20×**.

1. Amostra
2. Modelo e validação (principal + sensibilidade ao esquema de validação)
3. Figuras de desempenho (boxplot, matriz de confusão, ROC)
4. Interpretação — features, effect sizes, SHAP
5. Sensibilidade à qualidade do rastreio
6. Modelo salvo + resumo para o manuscrito

Figuras e tabelas vão para `reports/ml/09_et_*`; o modelo para `models/gb_ensemble_bcea.pkl`.

In [2]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import mannwhitneyu

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))
from ensemble_gb import EnsembleGB, log_t

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import RobustScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold

DIR_PROCESSED = RAIZ / 'data' / 'processed'
DIR_INTERIM   = RAIZ / 'data' / 'interim'
DIR_FIG       = RAIZ / 'reports' / 'ml'
DIR_MODELS    = RAIZ / 'models'
DIR_FIG.mkdir(parents=True, exist_ok=True); DIR_MODELS.mkdir(parents=True, exist_ok=True)

# Mesma paleta do 04_ML_eeg
BLUE_DARK, BLUE_MED, BLUE_LIGHT = '#2C3E50', '#2E86C1', '#AED6F1'
RED_MED    = '#D62728'
GRID_GRAY  = '#B0B0B0'
SPINE_GRAY = '#DDDDDD'
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif',
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'axes.edgecolor': SPINE_GRAY})

ModuleNotFoundError: No module named 'ensemble_gb'

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.std(ddof=1) ** 2 + (nb - 1) * b.std(ddof=1) ** 2) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else 0.0

def prettify(name):
    s = name
    s = s.replace('_Delta_FelizRaiva', ' Δ (Feliz−Raiva)').replace('_Delta_NeutroRaiva', ' Δ (Neutro−Raiva)')
    s = s.replace('_Delta_FelizNeutro', ' Δ (Feliz−Neutro)')
    s = s.replace('_Feliz', ' (Feliz)').replace('_Neutro', ' (Neutro)').replace('_Raiva', ' (Raiva)')
    repl = {'GazeX': 'Gaze X', 'GazeY': 'Gaze Y', 'Std': 'DP', 'Var': 'Var', 'Media': 'Média',
            'Mediana': 'Mediana', 'Perc': '%', 'PrimFix': '1ª Fixação', 'Scan': 'Scanpath',
            'Duracao': 'Duração', 'Fix': 'Fix', 'AOI': 'AOI', 'Geral': 'Geral', 'Olhos': 'Olhos',
            'Nariz': 'Nariz', 'Boca': 'Boca', 'Fora': 'Fora', 'Trans': 'Transição', 'Distancia': 'Distância',
            'Total': 'Total', 'Velocidade': 'Velocidade', 'Pupila': 'Pupila', 'Entropia': 'Entropia',
            'Revisitas': 'Revisitas', 'Latencia': 'Latência', 'BCEA': 'BCEA', 'ms': 'ms', 'Max': 'Máx',
            'Min': 'Mín', 'N': 'N'}
    return ' '.join(' '.join(repl.get(p, p) for p in s.split('_')).split())

## 1. Amostra

In [ ]:
META = ('ID_Sujeito', 'Grupo', 'Target')
df_et = pd.read_csv(DIR_PROCESSED / 'dataset_features_et.csv')
FEATS_FINAL = [c for c in df_et.columns if c not in META]
X_final  = df_et[FEATS_FINAL].fillna(0).values
y        = df_et['Target'].values
n        = len(y)
sujeitos = df_et['ID_Sujeito'].values
print(f'N = {n}  |  TEA = {(y==1).sum()}  |  Controle = {(y==0).sum()}  |  features = {len(FEATS_FINAL)}')

qual = pd.read_csv(DIR_INTERIM / 'qualidade_sujeitos.csv')
qual['ID_Sujeito'] = qual['Arquivo'].str.replace('.csv', '_ON.csv', regex=False)
print('\nGravações excluídas antes das features:')
print(qual.loc[qual['EXCLUIR'], ['Arquivo', 'Grupo', 'motivo']].to_string(index=False))
print('\nMantidas com alerta de qualidade:')
print(qual.loc[~qual['EXCLUIR'] & qual['alerta'].notna(),
               ['Arquivo', 'Grupo', 'taxa_rastreio', 'alerta']].round(1).to_string(index=False))

## 2. Modelo e validação

**Modelo:** média de P(TEA) de 360 Gradient Boosting de uma grade fixa (`src/ensemble_gb.py`) —
k ∈ {5,8,10,15,20} features pelo F da ANOVA × profundidade ∈ {1,2,3} × learning rate ∈
{0.03,0.1,0.3} × subsample ∈ {1.0,0.7} × árvores ∈ {50,100,200,400}. Nenhuma configuração é
escolhida pelos dados, então não há viés de seleção de modelo. Escala, filtro de variância e
ranking F são ajustados só no treino de cada fold.

**Validação principal:** 10-fold estratificado repetido 20× (20 sorteios dos folds). Cada
repetição dá uma predição fora-da-amostra para cada participante; as métricas são média ± DP
entre repetições.

**Sensibilidade ao esquema:** o mesmo ensemble em LOOCV e em 5-fold repetido, e uma GB única
(k=10, 200 árvores, lr 0.1, profundidade 2) nos três esquemas. Com N=41 o número depende do
esquema: o LOOCV é o extremo otimista e o 5-fold o pessimista.

> ⏱️ ~10 min.

In [3]:
def make_pipe(clf, k, n_feats):
    return Pipeline([('log', FunctionTransformer(log_t)),
                     ('sc',  RobustScaler()),
                     ('var', VarianceThreshold(0.0)),
                     ('sel', SelectKBest(f_classif, k=min(k, n_feats))),
                     ('clf', clf)])

N_FOLDS, N_REP = 10, 20

def gb_unico():   # o modelo anterior (uma GB só), como comparador
    return make_pipe(GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=2,
                                                random_state=42), 10, X_final.shape[1])

def _fit_pred(make, tr, te):
    return make().fit(X_final[tr], y[tr]).predict_proba(X_final[te])[:, 1]

def cv_repetido(make, n_folds, n_rep):
    """P(TEA) fora-da-amostra de cada sujeito; uma linha por repetição. n_folds=None -> LOOCV."""
    tarefas = [(r, tr, te) for r in range(n_rep)
               for tr, te in (LeaveOneOut().split(y) if n_folds is None else
                              StratifiedKFold(n_folds, shuffle=True, random_state=r).split(y, y))]
    res = joblib.Parallel(n_jobs=-1)(joblib.delayed(_fit_pred)(make, tr, te) for _, tr, te in tarefas)
    P = np.zeros((n_rep, n))
    for (r, _, te), p in zip(tarefas, res):
        P[r, te] = p
    return P

def metricas(P):
    rows = []
    for r, p in enumerate(P):
        pr = (p > 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pr).ravel()
        rows.append(dict(seed=r, accuracy=accuracy_score(y, pr), balanced_accuracy=balanced_accuracy_score(y, pr),
                         sensitivity=tp / (tp + fn), specificity=tn / (tn + fp), roc_auc=roc_auc_score(y, p)))
    return pd.DataFrame(rows)

t0 = time.time()
ESQUEMAS = {'10-fold × 20 (principal)': (N_FOLDS, N_REP), '5-fold × 20': (5, N_REP), 'LOOCV': (None, 1)}
P_cv = {(m, e): cv_repetido(mk, *args) for e, args in ESQUEMAS.items()
        for m, mk in (('Ensemble GB', EnsembleGB), ('GB única', gb_unico))}
tab_esquemas = pd.DataFrame([dict(modelo=m, esquema=e, acc=d.accuracy.mean(), acc_std=d.accuracy.std(),
                                  acc_min=d.accuracy.min(), acc_max=d.accuracy.max(),
                                  bal_acc=d.balanced_accuracy.mean(), sens=d.sensitivity.mean(),
                                  spec=d.specificity.mean(), roc_auc=d.roc_auc.mean())
                             for (m, e), d in ((k, metricas(P)) for k, P in P_cv.items())])
print(f'({time.time()-t0:.0f}s)')
display(tab_esquemas.round(4))

P_final    = P_cv[('Ensemble GB', '10-fold × 20 (principal)')]
df_metrics = metricas(P_final)
df_preds = pd.DataFrame([dict(seed=r, subject=sujeitos[i], y_true=int(y[i]), y_pred=int(P_final[r, i] > 0.5),
                              proba_asd=float(P_final[r, i])) for r in range(N_REP) for i in range(n)])
df_metrics.to_csv(DIR_FIG / '09_et_cv10_metrics_per_rep.csv', index=False)
df_preds.to_csv(DIR_FIG / '09_et_cv10_predictions.csv', index=False)

print(f'\nModelo final (ensemble GB) — média ± DP entre as {N_REP} repetições do {N_FOLDS}-fold:')
for c in ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc']:
    print(f'  {c:18} {df_metrics[c].mean():.4f} ± {df_metrics[c].std():.4f}')
df_metrics

NameError: name 'EnsembleGB' is not defined

## 3. Figuras de desempenho

### Variabilidade entre as 20 repetições

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
cm_cols = ['roc_auc', 'balanced_accuracy', 'sensitivity', 'specificity']
labels  = ['ROC-AUC', 'Balanced Acc', 'Sensibilidade', 'Especificidade']
data = [df_metrics[c].values for c in cm_cols]
ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.5,
           medianprops=dict(color=BLUE_DARK, linewidth=2),
           boxprops=dict(facecolor=BLUE_LIGHT, edgecolor=BLUE_MED),
           whiskerprops=dict(color=BLUE_MED), capprops=dict(color=BLUE_MED))
rng = np.random.default_rng(0)
for i, d in enumerate(data, start=1):
    ax.scatter(np.full(len(d), i) + rng.normal(0, 0.045, len(d)), d,
               color=BLUE_MED, alpha=0.85, s=28, zorder=3)
ax.grid(axis='y', linestyle='--', alpha=0.5, color=GRID_GRAY)
ax.set_ylabel('Score'); ax.set_ylim(0.55, 1.02)
ax.set_title(f'Variabilidade da performance ({N_FOLDS}-fold estratificado — {N_REP} repetições)', fontsize=13)
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_cv_boxplot.png', dpi=150, bbox_inches='tight'); plt.show()

### Matriz de confusão (repetição representativa = a de acurácia mais próxima da média)

In [ ]:
seed_repr = int((df_metrics.accuracy - df_metrics.accuracy.mean()).abs().idxmin())
sub = df_preds[df_preds.seed == seed_repr]
cm = confusion_matrix(sub.y_true, sub.y_pred)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.imshow(cm, cmap='Blues', vmin=0, vmax=cm.max())
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, fontweight='bold',
                color='white' if cm[i, j] > cm.max() * 0.55 else BLUE_DARK)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Controle', 'TEA']); ax.set_yticklabels(['Controle', 'TEA'])
ax.set_xlabel('Predito'); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusão agregada\n({N_FOLDS}-fold, repetição representativa)', fontsize=13)
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

### Curva ROC média

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
mean_fpr = np.linspace(0, 1, 100)
tprs, aucs = [], []
for sd in df_metrics.seed:
    s = df_preds[df_preds.seed == sd]
    fpr, tpr, _ = roc_curve(s.y_true, s.proba_asd)
    it = np.interp(mean_fpr, fpr, tpr); it[0] = 0.0
    tprs.append(it); aucs.append(roc_auc_score(s.y_true, s.proba_asd))
mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
std_tpr = np.std(tprs, axis=0)
ax.plot(mean_fpr, mean_tpr, color=BLUE_MED, lw=3,
        label=f'ROC média (AUC = {np.mean(aucs):.2f} ± {np.std(aucs):.2f})')
ax.fill_between(mean_fpr, np.clip(mean_tpr - std_tpr, 0, 1), np.clip(mean_tpr + std_tpr, 0, 1),
                color=BLUE_LIGHT, alpha=0.3, label='± 1 DP')
ax.plot([0, 1], [0, 1], ls='--', lw=2, color='gray', alpha=0.8, label='Acaso')
ax.grid(linestyle='--', alpha=0.5, color=GRID_GRAY)
ax.set_xlabel('Falso positivo'); ax.set_ylabel('Verdadeiro positivo')
ax.set_title(f'Curva ROC ({N_FOLDS}-fold estratificado — {N_REP} repetições)', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_roc_curve.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. Interpretação — refit nos 41 participantes

O ensemble usa as 20 features de maior F: as 5 primeiras entram em todos os 360 membros, as 10
primeiras em ≥ 60%, e as de posição 11–20 em 20–40%. A tabela traz, para cada uma, Cohen's *d* e
Mann-Whitney (TEA vs Controle).

In [ ]:
ens_final = EnsembleGB().fit(X_final, y)
cols_ens = [FEATS_FINAL[i] for i in ens_final.colunas_usadas()]          # 20 features, em ordem de F
selected = cols_ens[:10]                                                 # as 10 primeiras (≥ 60% dos membros)
imp_ens  = ens_final.importancias()
f_ens    = ens_final.f_[ens_final.ordem_[:len(cols_ens)]]
freq_ens = np.array([np.mean([k >= pos + 1 for k in ens_final.ks]) for pos in range(len(cols_ens))])
acc_resub = accuracy_score(y, ens_final.predict(X_final))  # só sanity check, NÃO é validação

tab_ens = pd.DataFrame([dict(feature=f, frac_membros=fr, importancia=im, F=fs,
                             d=cohens_d(df_et[f].fillna(0).values[y == 1], df_et[f].fillna(0).values[y == 0]),
                             p_MW=mannwhitneyu(df_et[f].fillna(0).values[y == 1],
                                               df_et[f].fillna(0).values[y == 0]).pvalue)
                        for f, fr, im, fs in zip(cols_ens, freq_ens, imp_ens, f_ens)])
print(f'{len(cols_ens)} features do ensemble (refit nos {n} sujeitos), em ordem de F:')
display(tab_ens.round(4))

### Effect size (Cohen's d) das 10 features mais bem ranqueadas

In [ ]:
df_d = (pd.DataFrame([(prettify(f), cohens_d(df_et[f].fillna(0).values[y == 1],
                                             df_et[f].fillna(0).values[y == 0])) for f in selected],
                     columns=['Feature', 'CohensD'])
          .sort_values('CohensD'))
fig, ax = plt.subplots(figsize=(8, 5.5))
norm = TwoSlopeNorm(vmin=min(-1, df_d.CohensD.min() - 0.1), vcenter=0, vmax=max(1, df_d.CohensD.max() + 0.1))
cmap = plt.get_cmap('RdBu_r')
ax.barh(df_d.Feature, df_d.CohensD, color=[cmap(norm(v)) for v in df_d.CohensD], edgecolor='white')
ax.axvline(0, color='#444', lw=1)
ax.set_xlabel("Cohen's d (TEA − Controle)")
ax.set_title('Effect size das 10 features mais bem ranqueadas', fontsize=13)
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, pad=0.02).set_label("Cohen's d")
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_effect_sizes.png', dpi=150, bbox_inches='tight'); plt.show()

### Importância média no ensemble e F-score das 20 features

In [ ]:
import matplotlib.patches as mpatches
df_feat = (pd.DataFrame({'Feature': [prettify(f) for f in cols_ens], 'GB': imp_ens, 'F': f_ens,
                         'BCEA': [f.startswith('BCEA_') for f in cols_ens]})
             .sort_values('GB'))
cor = df_feat['BCEA'].map({True: RED_MED, False: BLUE_MED})
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Features do ensemble de GB (20 de maior F)', fontsize=13, fontweight='bold')
for ax, col, lab in zip(axes, ['GB', 'F'], ['Importância média no ensemble', 'F-score (ANOVA)']):
    ax.barh(df_feat['Feature'], df_feat[col], color=list(cor), edgecolor='white')
    ax.set_xlabel(lab)
axes[1].set_yticklabels([])
axes[0].legend(handles=[mpatches.Patch(color=RED_MED, label='BCEA'),
                        mpatches.Patch(color=BLUE_MED, label='outra')], fontsize=10, loc='lower right')
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

### SHAP (média entre os 360 membros; 10 features de maior |SHAP|)

In [ ]:
import shap
shap_values, Zs = ens_final.shap_values(X_final)
Xs_df = pd.DataFrame(Zs, columns=[prettify(c) for c in cols_ens])
shap_rank = (pd.Series(np.abs(shap_values).mean(0), index=cols_ens).sort_values(ascending=False))
print(shap_rank.head(10).round(4).to_string())
plt.figure(figsize=(8, 5.5))
shap.summary_plot(shap_values, Xs_df, show=False, plot_size=None, max_display=10)
plt.gcf().suptitle('SHAP — modelo unimodal eye-tracking (ensemble de GB)', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig(DIR_FIG / '09_et_shap_importance.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. Sensibilidade à qualidade do rastreio

A dispersão do olhar aumenta quando o rastreio é instável, e o grupo TEA rastreia pior. Dois
testes: (a) quanto a **taxa de rastreio sozinha** separa os grupos; (b) o mesmo modelo e a mesma
validação **sem** os participantes com alerta de rastreio insuficiente ou calibração deslocada.

In [ ]:
q = qual.set_index('ID_Sujeito').loc[sujeitos]
auc_rastreio = roc_auc_score(y, -q['taxa_rastreio'].values)
print(f'ROC-AUC da taxa de rastreio sozinha (menor rastreio -> TEA): {auc_rastreio:.3f}')

alerta = q['alerta'].fillna('')
fora = (alerta.str.contains('rastreio insuficiente') | alerta.str.contains('calibração deslocada')).values
print(f'\nRetirados ({fora.sum()}):', ', '.join(s.replace('_ON.csv', '') for s in sujeitos[fora]))

_X, _y, _n = X_final, y, n                      # cv_repetido/metricas usam X_final, y, n
X_final, y, n = _X[~fora], _y[~fora], int((~fora).sum())
df_sens = metricas(cv_repetido(EnsembleGB, N_FOLDS, N_REP))
n_sens, n_sens_tea = n, int((y == 1).sum())
X_final, y, n = _X, _y, _n

print(f'\nEnsemble GB, {N_FOLDS}-fold × {N_REP}, N = {n_sens} (TEA = {n_sens_tea}):')
for c in ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc']:
    print(f'  {c:18} {df_sens[c].mean():.4f} ± {df_sens[c].std():.4f}')

## 6. Modelo salvo + resumo (para o texto do manuscrito)

In [4]:
joblib.dump(ens_final, DIR_MODELS / 'gb_ensemble_bcea.pkl')

def _ms(d, c): return {'mean': float(d[c].mean()), 'std': float(d[c].std())}
summary = {
    'n_subjects': int(n), 'n_asd': int((y == 1).sum()), 'n_control': int((y == 0).sum()),
    'modelo': 'Ensemble GB (media de 360 GB; src/ensemble_gb.py) sobre 252 features (emocoes + avancadas, com BCEA)',
    'validacao': f'{N_FOLDS}-fold estratificado repetido {N_REP}x (media +- DP entre repeticoes)',
    'n_repeticoes': len(df_metrics),
    'accuracy_mean': float(df_metrics.accuracy.mean()), 'accuracy_std': float(df_metrics.accuracy.std()),
    'accuracy_min': float(df_metrics.accuracy.min()), 'accuracy_max': float(df_metrics.accuracy.max()),
    'balanced_accuracy_mean': float(df_metrics.balanced_accuracy.mean()),
    'balanced_accuracy_std': float(df_metrics.balanced_accuracy.std()),
    'sensitivity_mean': float(df_metrics.sensitivity.mean()), 'sensitivity_std': float(df_metrics.sensitivity.std()),
    'specificity_mean': float(df_metrics.specificity.mean()), 'specificity_std': float(df_metrics.specificity.std()),
    'roc_auc_mean': float(df_metrics.roc_auc.mean()), 'roc_auc_std': float(df_metrics.roc_auc.std()),
    'representative_rep': seed_repr,
    'representative_rep_confusion_matrix': cm.tolist(),
    'por_esquema': tab_esquemas.round(4).to_dict('records'),
    'features_ensemble': cols_ens,
    'features_top10': selected,
    'univariado_top20': tab_ens.round(4).to_dict('records'),
    'shap_top10': list(shap_rank.index[:10]),
    'sensibilidade_rastreio': {
        'auc_taxa_rastreio_sozinha': float(auc_rastreio),
        'retirados': [s.replace('_ON.csv', '') for s in sujeitos[fora]],
        'n': n_sens, 'n_asd': n_sens_tea,
        **{c: _ms(df_sens, c) for c in ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'roc_auc']}},
    'resubstitution_accuracy_final_refit': float(acc_resub),
    'nota': ('O numero depende do esquema de validacao (ver por_esquema): LOOCV e o extremo otimista, '
             '5-fold o pessimista. resubstitution_accuracy_final_refit e apenas sanity check.'),
}
(DIR_FIG / '09_et_results_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))

meta = {
    'n_sujeitos': int(n), 'n_tea': int((y == 1).sum()), 'n_controle': int((y == 0).sum()),
    'features_entrada': FEATS_FINAL,
    'features_ensemble': cols_ens,
    'grade': {'k_features': list(ens_final.ks), 'max_depth': list(ens_final.depths),
              'learning_rate': list(ens_final.lrs), 'subsample': list(ens_final.subsamples),
              'n_estimators': list(ens_final.n_estimators)},
    'cv10_repetido_accuracy': float(df_metrics.accuracy.mean()),
    'cv10_repetido_accuracy_std': float(df_metrics.accuracy.std()),
    'cv10_repetido_balanced_acc': float(df_metrics.balanced_accuracy.mean()),
    'n_repeticoes': len(df_metrics),
}
(DIR_MODELS / 'gb_ensemble_bcea_meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in summary.items()
                  if k not in ('features_top10', 'features_ensemble', 'por_esquema', 'univariado_top20',
                               'representative_rep_confusion_matrix')}, indent=2, ensure_ascii=False))

NameError: name 'ens_final' is not defined